## 1. Load parametric description of the line

In [ ]:
import numpy
import sys
import meshio
import vtk
from scipy.interpolate import BSpline
from scipy.spatial import cKDTree

sys.path.insert(0, "../src")  # add Folder_2 path to search list

from meshing_functions import export_centerline, export_mask

h = 0.9
id = 7
extension_factor = 0.08


pixel_spacing = numpy.loadtxt("/Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/VTI_0.5_fat_AIMax/pixel_spacing.txt")[0]
offset = numpy.loadtxt("/Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/cropped T1/Meshes/offset.txt")

base_centerline = "/Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/cropped T1/Meshes/centerlines/"
base_mesh = "/Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/cropped T1/Meshes/re-meshed_h_"+str(h)+"/"


data = numpy.load(base_centerline + "centerline_id_"+str(id)+"_parametric.npz")

sx = BSpline(data["tx"], data["cx"], data["kx"])
sy = BSpline(data["ty"], data["cy"], data["ky"])
sz = BSpline(data["tz"], data["cz"], data["kz"])

mesh = meshio.read(base_mesh + "id_"+str(id)+"__h_"+str(h)+"_snapped.stl")
points = (mesh.points + offset)/pixel_spacing

### Normals just from the surface mesh

In [219]:
triangles = mesh.cells_dict["triangle"]

normals = numpy.zeros_like(points)

for tri in triangles:
    
    p0 = points[tri[0]]
    p1 = points[tri[1]]
    p2 = points[tri[2]]

    n = numpy.cross(p1 - p0, p2 - p0)

    normals[tri] += n

N = normals / numpy.linalg.norm(normals, axis=1)[:,None]



### Sample centerline

In [220]:
# spline domain
k = int(data["kx"])
tx = data["tx"]

tmin = tx[k]
tmax = tx[-k-1]

ts = numpy.linspace(tmin, tmax, 1000)

curve = numpy.vstack([
    sx(ts),
    sy(ts),
    sz(ts)
]).T
# build spatial search structure
tree = cKDTree(curve)


print(tmin, tmax)

-33.430410106526864 34.65627785031621


### Spline derivatives

In [221]:
dsx = sx.derivative()
dsy = sy.derivative()
dsz = sz.derivative()



### Find closest centerline location & compute directions

In [222]:
# find nearest centerline point for every mesh vertex
dist, idx = tree.query(points)


# corresponding spline parameter for each vertex
t_closest = ts[idx]

# compute centerline tangent vectors
Tx = dsx(t_closest)
Ty = dsy(t_closest)
Tz = dsz(t_closest)

# tangent direction
T = numpy.vstack([Tx, Ty, Tz]).T

# circumferential direction
C = numpy.cross(T, N)

# normalize vectors
T = T / numpy.linalg.norm(T, axis=1)[:, None]
C = C / numpy.linalg.norm(C, axis=1)[:,None]


### Find endpoints

In [ ]:
# evaluate endpoints
dt = extension_factor * (tmax - tmin)   # % extension

end1 = numpy.array([sx(tmin - dt), sy(tmin - dt), sz(tmin - dt)])
end2 = numpy.array([sx(tmax + dt), sy(tmax + dt), sz(tmax + dt)])

print("End 1:", end1)
print("End 2:", end2)

tree_mesh = cKDTree(points)

# find closest nodes
dist1, idx1 = tree_mesh.query(end1)
dist2, idx2 = tree_mesh.query(end2)

print("End 1 node index:", idx1, "distance:", dist1)
print("End 2 node index:", idx2, "distance:", dist2)

end1_node = mesh.points[idx1]
end2_node = mesh.points[idx2]

ends_flag = numpy.zeros(points.shape)
ends_flag[idx1] = 1
ends_flag[idx2] = 1



End 1: [ 69.82406911  59.63909656 126.32698249]
End 2: [102.09566398 131.75338248 101.68991073]
End 1 node index: 537 distance: 2.597613236819185
End 2 node index: 149 distance: 0.7572827241187519


In [224]:
mesh.point_data["centerline_direction"] = T
mesh.point_data["normal_direction"] = N
mesh.point_data["circumferential_direction"] = C
mesh.point_data["end_point_flag"] = ends_flag

meshio.write(
    base_mesh + "id_"+str(id)+"__h_"+str(h)+"_snapped-with_direction.vtu",
    mesh
)


numpy.savetxt(
    base_mesh + "end_point_id_"+str(id)+".csv",
    [
        [idx1, end1_node[0], end1_node[1], end1_node[2]],
        [idx2, end2_node[0], end2_node[1], end2_node[2]]
    ],
    fmt=["%d", "%.6f", "%.6f", "%.6f"],
    delimiter=",",
    header="id,x,y,z",
    comments=""
)

data = numpy.column_stack([
    numpy.arange(len(points)),   # node IDs
    mesh.points,                   # x,y,z
    T                         # Tx,Ty,Tz
])

numpy.savetxt(
    base_mesh + "tangent_vectors_id_"+str(id)+".csv",
    data,
    fmt=["%d","%.6f","%.6f","%.6f","%.6f","%.6f","%.6f"],
    delimiter=",",
    header="id,x,y,z,Tx,Ty,Tz",
    comments=""
)

In [225]:
print("t range:", tmin, tmax)
print("curve min:", curve.min(axis=0))
print("curve max:", curve.max(axis=0))
print("mesh min:", mesh.points.min(axis=0))
print("mesh max:", mesh.points.max(axis=0))

t range: -33.430410106526864 34.65627785031621
curve min: [ 71.83843599  67.83858239 106.09099573]
curve max: [ 97.26928259 127.61213117 127.10587455]
mesh min: [-2.1081342   2.47236913  4.89946441]
mesh max: [11.18605976 24.79337525 14.6702512 ]
